# Customer Explainability — CustomerInsight AI

## Research objective

Investigate how the four baseline K-Means customer groups differ
in their original Recency, Frequency and Monetary (RFM) values.

**Research question (RQ4):** How can the resulting customer groups
be described using transparent purchasing statistics?

This notebook reconstructs the customer profiles and compares
them with the saved Phase 06 research evidence.


## 1. Environment and research files

We reuse the existing clustering implementation.

No algorithms are reimplemented from scratch.
The notebook does not overwrite the original experimental CSVs.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if not (ROOT / "src/customerinsight").is_dir():
    raise FileNotFoundError("Run this notebook from the project or notebooks folder.")

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from customerinsight.clustering import (
    prepare_clustering_features,
    run_clustering_experiments,
)

RFM_FILE = ROOT / "data/processed/customer_rfm.csv"
PROFILES_FILE = ROOT / "results/experiments/customer_cluster_profiles.csv"

print("Project directory:", ROOT)


Project directory: C:\Users\x03xN\Documents\customerinsight-ai


## 2. Load saved customer data and profiles

The RFM table represents historical positive purchasing activity.

- **Recency:** Days since the last qualifying purchase
- **Frequency:** Number of distinct qualifying invoices
- **Monetary:** Total positive qualifying purchase value

The saved profile table is our reference evidence.


In [2]:
rfm = pd.read_csv(RFM_FILE, dtype={"customer_id": "string"})
saved_profiles = pd.read_csv(PROFILES_FILE)

required_rfm = {"customer_id", "recency", "frequency", "monetary"}
required_profiles = {
    "cluster", "customers", "median_recency",
    "median_frequency", "median_monetary"
}

assert required_rfm.issubset(rfm.columns)
assert required_profiles.issubset(saved_profiles.columns)
assert rfm["customer_id"].is_unique
assert not rfm.empty

print("Customers:", len(rfm))
print("Saved groups:", len(saved_profiles))

display(rfm.head())
display(saved_profiles)


Customers: 4338
Saved groups: 4


,customer_id,recency,frequency,monetary
0,12346,326,1,77183.60
1,12347,3,7,4310.00
2,12348,76,4,1797.24
3,12349,19,1,1757.55
4,12350,311,1,334.40


,cluster,customers,median_recency,median_frequency,median_monetary,total_positive_spending,customer_share_pct
0,0,877,19.0,2.0,450.84,462331.95,20.22
1,1,1579,183.0,1.0,300.42,556006.67,36.40
2,2,720,9.0,10.0,3726.08,5786787.92,16.60
3,3,1162,54.0,4.0,1353.74,2106281.36,26.79


## 3. Recreate the baseline cluster assignments

Use the same `log1p` preprocessing and baseline clustering
function used in Phase 04.

K-Means is configured in the existing project module with
four clusters, `random_state=42` and `n_init=10`.

Group numbers are arbitrary identifiers, not rankings.


In [3]:
features = prepare_clustering_features(rfm)
reproduced_results, labels_by_model = run_clustering_experiments(features)

labels = np.asarray(labels_by_model["K-Means"])

assert len(labels) == len(rfm)
assert len(set(labels)) == 4

customers = rfm.copy()
customers["cluster"] = labels

display(reproduced_results)
display(customers["cluster"].value_counts().sort_index())


,algorithm,clusters,silhouette,coverage
0,K-Means,4,0.332372,1.0000
1,Agglomerative,4,0.270554,1.0000
2,DBSCAN,2,0.289804,0.9818


cluster
0     877
1    1579
2     720
3    1162
Name: count, dtype: int64

## 4. Reconstruct group characteristics

Pandas aggregation is used to calculate the actual
number of customers and median RFM statistics.

We compare these with the saved research table to check
whether the group profiles can be reproduced.


In [4]:
recreated = (
    customers.groupby("cluster")
    .agg(
        customers=("customer_id", "size"),
        median_recency=("recency", "median"),
        median_frequency=("frequency", "median"),
        median_monetary=("monetary", "median"),
    )
    .reset_index()
)

recreated["customer_share_pct"] = (
    100 * recreated["customers"] / len(customers)
)

display(recreated)


,cluster,customers,median_recency,median_frequency,median_monetary,customer_share_pct
0,0,877,19.0,2.0,450.84,20.21669
1,1,1579,183.0,1.0,300.42,36.399262
2,2,720,9.0,10.0,3726.08,16.59751
3,3,1162,54.0,4.0,1353.74,26.786538


In [5]:
comparison = recreated.merge(
    saved_profiles[
        [
            "cluster",
            "customers",
            "median_recency",
            "median_frequency",
            "median_monetary",
        ]
    ],
    on="cluster",
    how="outer",
    suffixes=("_new", "_saved"),
    indicator=True,
)

measurements = [
    "customers",
    "median_recency",
    "median_frequency",
    "median_monetary",
]

comparison["profiles_match"] = comparison["_merge"].eq("both")

for measurement in measurements:
    new = comparison[f"{measurement}_new"]
    saved = comparison[f"{measurement}_saved"]

    comparison["profiles_match"] &= np.isclose(
        new, saved, atol=0.01, rtol=0, equal_nan=False
    )

display(comparison)

if comparison["profiles_match"].all():
    print("All four group profiles reproduced within tolerance.")
else:
    print(
        "Profile mismatch detected. Review cluster assignments, "
        "preprocessing and original profile-generation methods."
    )


,cluster,customers_new,median_recency_new,median_frequency_new,median_monetary_new,customer_share_pct,customers_saved,median_recency_saved,median_frequency_saved,median_monetary_saved,_merge,profiles_match
0,0,877,19.0,2.0,450.84,20.21669,877,19.0,2.0,450.84,both,True
1,1,1579,183.0,1.0,300.42,36.399262,1579,183.0,1.0,300.42,both,True
2,2,720,9.0,10.0,3726.08,16.59751,720,9.0,10.0,3726.08,both,True
3,3,1162,54.0,4.0,1353.74,26.786538,1162,54.0,4.0,1353.74,both,True


All four group profiles reproduced within tolerance.


## 5. Group distribution

Customer group sizes help identify whether the segmentation
represents several substantial groups or a highly imbalanced partition.

These four groups belong to the baseline solution;
they must not be confused with the separate K=2 diagnostic.


In [6]:
fig = px.bar(
    recreated,
    x="cluster",
    y="customers",
    text="customers",
    title="Number of customers in each baseline group",
    labels={"cluster": "Cluster", "customers": "Customers"},
)

fig.update_xaxes(type="category")
fig.show()


## 6. Compare median purchasing behaviour

The figures below use original RFM measurements,
not standardized feature values.

Lower recency means a more recent recorded purchase.
Frequency and spending reflect observed transaction history.


In [7]:
metrics = {
    "median_recency": "Median recency (days)",
    "median_frequency": "Median purchase frequency",
    "median_monetary": "Median positive spending (£)",
}

for column, label in metrics.items():
    fig = px.bar(
        recreated,
        x="cluster",
        y=column,
        text_auto=".2f",
        title=label,
        labels={"cluster": "Cluster", column: label},
    )
    fig.update_xaxes(type="category")
    fig.show()


## 7. Using our existing explainability module

The project also contains:

- `assign_customer_clusters`
- `describe_clusters`
- `compare_with_population`
- `generate_cluster_descriptions`

Their exact implementations should be inspected before
executing them here. This ensures we respect their expected
inputs and do not assume an undocumented return structure.


In [8]:
import inspect
from customerinsight import explanations

for name in [
    "assign_customer_clusters",
    "describe_clusters",
    "compare_with_population",
    "generate_cluster_descriptions",
]:
    function = getattr(explanations, name)
    print(f"{name}{inspect.signature(function)}")


assign_customer_clusters(rfm: pandas.DataFrame, clusters: int = 4) -> pandas.DataFrame
describe_clusters(customers: pandas.DataFrame) -> pandas.DataFrame
compare_with_population(customers: pandas.DataFrame) -> pandas.DataFrame
generate_cluster_descriptions(profiles: pandas.DataFrame) -> pandas.DataFrame


## 8. Interpretation

Using the saved baseline profiles:

- Cluster 0: relatively recent, lower-frequency purchasers.
- Cluster 1: less-recent, low-frequency purchasers.
- Cluster 2: recent, high-frequency and higher-spending purchasers.
- Cluster 3: customers with intermediate purchasing characteristics.

These statements describe group-level medians. They do not
establish loyalty, customer satisfaction, churn probability or
individual future behaviour.

Interpretation should be reviewed again if the reconstructed
profiles do not match the original saved evidence.


## 9. Limitations

1. RFM summarizes a limited set of purchasing behaviours.
2. Group medians can hide substantial within-group variation.
3. Group labels are descriptive, not verified customer categories.
4. Monetary values represent qualifying positive spending,
   not fully reconciled net profit.
5. Agreement with saved profiles does not establish business value.
6. This notebook uses the processed RFM data, not the original
   transaction workbook.

## 10. Reproducibility conclusion

Record the actual comparison outcome after executing the notebook.

A matching profile table supports reproducibility of the
baseline group summaries in the current environment.
It does not establish end-to-end data preparation reproducibility.
